# Clinical Trial GRPO Training - V3.1

**Key fixes over V3:**
1. **Multi-step rollout (5 steps)** - model controls step 0, smart continuation for 1-4. Reward range 0→10+
2. **JSON quality bonus** - +1.0 for valid parsed JSON, -0.5 for fallback
3. **Confidence capped at 0.7** - avoids binary overconfidence cliff at 0.8
4. **Same seed per group + parsed confidence** (from V3)
5. **More steps (100) + higher LR (2e-5)**

Run cells in order. Restart runtime after Cell 1 if prompted.

In [ ]:
# Cell 1: Install dependencies (restart runtime after this cell)
import shutil, os
cache_dir = os.path.join(os.path.expanduser('~'), '.cache', 'unsloth_compiled_cache')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f'Cleared stale cache: {cache_dir}')
else:
    print('No stale cache found')

!pip install -q --no-deps unsloth
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "trl>=0.15.2" peft transformers accelerate datasets
!pip install -q requests matplotlib huggingface_hub numpy

print('Install complete. Restart runtime now, then run Cell 2.')

In [ ]:
# Cell 2: Imports & config
import json, os, re, csv, random, requests, numpy as np
from datetime import datetime, timezone

ENV_URL = "https://roopalgn-openenv-clinical-trial.hf.space"
NUM_GENERATIONS = 4        # fewer gens (longer rollouts need more VRAM)
NUM_PROMPTS = 100
BASE_SEED = 42
MODEL_SIZE = "1.5b"
MAX_COMPLETION_LEN = 512   # room for multi-action plan
TEMPERATURE = 0.7
CONTINUATION_STEPS = 14    # 15 total steps to reach terminal rewards
CONFIDENCE_CAP = 0.7       # cap below 0.8 overconfidence threshold

SYSTEM_PROMPT = """You are a clinical trial designer.
Given the current trial state and available actions, choose the BEST next action.
Return exactly ONE valid JSON object:
{"action_type": "<from available_actions>", "parameters": {}, "justification": "why", "confidence": 0.5}
Do not add any text outside the JSON."""

print(f"Config: {MODEL_SIZE} model, {NUM_GENERATIONS} gen, {NUM_PROMPTS} prompts, {CONTINUATION_STEPS} cont steps")

In [ ]:
# Cell 3: Environment helpers

def env_reset(seed=None):
    payload = {"seed": seed} if seed is not None else {}
    resp = requests.post(f"{ENV_URL}/reset", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()


def env_step(action_type, parameters=None, justification="", confidence=0.3):
    """Step the environment. Default confidence=0.3 (safe from overconfidence)."""
    payload = {
        "action_type": action_type,
        "parameters": parameters or {},
        "justification": justification,
        "confidence": confidence,
    }
    resp = requests.post(f"{ENV_URL}/step", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()


def parse_action(text, available_actions=None):
    """Extract JSON action + confidence from model output."""
    candidates = []
    import re as _re
    fenced = _re.findall(r"```json\s*(.*?)\s*```", text, flags=_re.DOTALL | _re.IGNORECASE)
    candidates.extend(fenced)
    candidates.append(text)

    for candidate in candidates:
        start = candidate.find("{")
        end = candidate.rfind("}")
        if start == -1 or end == -1 or end <= start:
            continue
        try:
            parsed = json.loads(candidate[start:end + 1])
            at = str(parsed.get("action_type", "")).strip()
            params = parsed.get("parameters", {})
            if not isinstance(params, dict):
                params = {}
            try:
                conf = float(parsed.get("confidence", 0.3))
                conf = min(max(conf, 0.0), 1.0)
            except (TypeError, ValueError):
                conf = 0.3
            justification = str(parsed.get("justification", ""))
            if available_actions and at not in available_actions:
                at = available_actions[0]
            elif not at:
                at = "set_primary_endpoint"
            return {"action_type": at, "parameters": params,
                    "confidence": conf, "justification": justification}
        except Exception:
            continue

    # Fallback: pick first available action, LOW confidence
    if available_actions:
        return {"action_type": available_actions[0], "parameters": {},
                "confidence": 0.3, "justification": "fallback"}
    return {"action_type": "set_primary_endpoint", "parameters": {},
            "confidence": 0.3, "justification": "fallback"}


def extract_reward(result):
    """Extract total reward from step result."""
    reward = result.get("reward", 0.0)
    if isinstance(reward, dict):
        return float(sum(float(v) for v in reward.values()))
    return float(reward)


def multi_step_reward(model_response, seed, continuation_steps=CONTINUATION_STEPS):
    """Score a model output via multi-step rollout.

    Step 0: model's parsed action. Steps 1-N: smart continuation.
    Adds JSON quality bonus and caps confidence.
    Reward range: ~0 to 10+ (vs single-step 0 to 1.2).
    """
    try:
        obs = env_reset(seed=seed)
        available = obs.get("available_actions", ["set_primary_endpoint"])
        action = parse_action(model_response, available)

        # JSON quality bonus: +1.0 for valid parse, -0.5 for garbage
        is_parsed = action.get("justification", "") != "fallback"
        json_bonus = 1.0 if is_parsed else -0.5

        # Cap confidence to avoid the 0.8 overconfidence cliff
        confidence = min(action["confidence"], CONFIDENCE_CAP)

        # Step 0: model's action
        result = env_step(
            action["action_type"],
            action.get("parameters", {}),
            justification=action.get("justification", ""),
            confidence=confidence,
        )
        total_reward = extract_reward(result)

        # Steps 1-N: smart continuation with varied actions
        for i in range(continuation_steps):
            if result.get("done", False):
                break
            avail = result.get("observation", {}).get("available_actions", [])
            if not avail:
                break
            act = avail[i % len(avail)]
            result = env_step(act, confidence=0.3)
            total_reward += extract_reward(result)

        return total_reward + json_bonus
    except Exception as e:
        print(f"Reward error: {e}")
        return -2.0


def build_prompt(obs):
    """Build observation-rich prompt from env state."""
    available = obs.get("available_actions", ["set_primary_endpoint"])
    phase = obs.get("phase_data", {}).get("current_phase", "unknown")
    return (
        f"You are designing a clinical trial.\n\n"
        f"Scenario: {obs.get('scenario_description', '')}\n"
        f"Current phase: {phase}\n"
        f"Resources: {json.dumps(obs.get('resource_status', {}))}\n"
        f"Available actions: {available}\n"
        f"Steps taken: {obs.get('steps_taken', 0)}/{obs.get('max_steps', 100)}\n"
        f"Hint: {obs.get('hint', '')}\n\n"
        f"Choose ONE action from: {available}\n"
        f"Return ONLY JSON: "
        '{"action_type": "<from list above>", "parameters": {}, '
        '"justification": "...", "confidence": 0.5}'
    )

print("Helpers defined.")


In [ ]:
# Cell 4: Connection test + dry-run validation
print(f"Testing connection to {ENV_URL}...")
try:
    r = requests.get(f"{ENV_URL}/ping", timeout=10)
    print(f"Ping: {r.json()}")
except Exception as e:
    raise RuntimeError(f"Cannot connect to env: {e}. Is the HF Space running?") from e

# Dry run: verify reward VARIANCE with multi-step rollout
print("\n=== Dry Run: Multi-step reward discrimination ===")
test_seed = 42
obs = env_reset(seed=test_seed)
available = obs.get("available_actions", ["set_primary_endpoint"])
print(f"Available actions at step 0 for seed {test_seed}: {available}")
print(f"(Only {len(available)} actions - multi-step + JSON bonus create variance)")

# Test multi-step with each first action
test_rewards = []
for action_name in available:
    fake_json = json.dumps({"action_type": action_name, "parameters": {}, "justification": "test", "confidence": 0.5})
    r = multi_step_reward(fake_json, test_seed)
    test_rewards.append(r)
    print(f"  First={action_name:35s} -> multi-step reward = {r:.4f}")

# Test: valid JSON vs garbage (shows JSON bonus effect)
print("\n=== JSON quality variance test ===")
valid_json = json.dumps({"action_type": available[0], "parameters": {}, "justification": "good", "confidence": 0.5})
good_r = multi_step_reward(valid_json, test_seed)
garbage_r = multi_step_reward("I think we should do some analysis maybe", test_seed)
print(f"  Valid JSON   -> {good_r:.4f}")
print(f"  Garbage text -> {garbage_r:.4f}")
print(f"  Delta: {good_r - garbage_r:.4f} (JSON quality bonus effect)")
test_rewards.extend([good_r, garbage_r])

reward_std = float(np.std(test_rewards))
print(f"\nReward std across test cases: {reward_std:.4f}")
if reward_std > 0.1:
    print("GOOD: sufficient variance for GRPO.")
else:
    print("WARNING: Reward std too low.")


In [ ]:
# Cell 5: Generate diverse prompts
print(f"Generating {NUM_PROMPTS} diverse prompts...")
prompts = []
prompt_seeds = []
for i in range(NUM_PROMPTS):
    seed = BASE_SEED + i
    try:
        obs = env_reset(seed=seed)
        prompt_text = build_prompt(obs)
        prompts.append(prompt_text)
        prompt_seeds.append(seed)
    except Exception as e:
        print(f"  Prompt gen failed for seed {seed}: {e}")

print(f"Generated {len(prompts)} prompts from {len(set(prompt_seeds))} unique seeds")
if len(prompts) < 5:
    raise RuntimeError("Too few prompts generated. Check HF Space.")


In [ ]:
# Cell 6: Load model
from unsloth import FastLanguageModel

MODEL_PRESETS = {
    "1.5b": {"model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", "lora_r": 8, "seq_len": 2048},
    "3b":   {"model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",   "lora_r": 16, "seq_len": 3072},
    "7b":   {"model_name": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",   "lora_r": 32, "seq_len": 4096},
}
preset = MODEL_PRESETS[MODEL_SIZE]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=preset["model_name"],
    max_seq_length=preset["seq_len"],
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=preset["lora_r"],
    lora_alpha=preset["lora_r"] * 2,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {preset['model_name']}")
model.print_trainable_parameters()


In [ ]:
# Cell 7: Build dataset + reward function + train
from datasets import Dataset
import torch
from trl import GRPOConfig, GRPOTrainer

# Build chat-formatted dataset
chat_prompts = []
for idx, prompt_text in enumerate(prompts):
    chat_prompts.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text},
        ],
    })
train_dataset = Dataset.from_list(chat_prompts)

# ============================================================
# REWARD FUNCTION - V3.1 core
# All completions in a GRPO group use the SAME seed (same env).
# Multi-step rollout + JSON quality bonus + confidence cap.
# ============================================================
_call_counter = [0]

def reward_func(completions, **kwargs):
    """Score a batch of completions. All share the SAME env seed."""
    prompt_idx = _call_counter[0] // NUM_GENERATIONS
    group_seed = prompt_seeds[prompt_idx % len(prompt_seeds)]

    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        else:
            text = str(completion)
        reward = multi_step_reward(text, group_seed)
        rewards.append(reward)
        _call_counter[0] += 1

    return rewards

# Auto precision for T4 compatibility
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Precision: bf16={use_bf16}, fp16={use_fp16}")

training_args = GRPOConfig(
    output_dir="checkpoints/grpo_v3",
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LEN,
    temperature=TEMPERATURE,
    learning_rate=2e-5,
    num_train_epochs=1,
    per_device_train_batch_size=NUM_GENERATIONS,
    gradient_accumulation_steps=1,
    max_steps=NUM_PROMPTS,
    warmup_steps=5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=1,
    save_steps=max(1, NUM_PROMPTS // 4),
    report_to="none",
    bf16=use_bf16,
    fp16=use_fp16,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    reward_funcs=[reward_func],
)

print(f"\nTraining: {NUM_PROMPTS} steps x {NUM_GENERATIONS} gen/step, LR=2e-5")
print(f"Multi-step rollout: {1 + CONTINUATION_STEPS} steps per completion")
print(f"Confidence cap: {CONFIDENCE_CAP} (below 0.8 overconfidence threshold)")
print("Starting...\n")

start_time = datetime.now(timezone.utc)
trainer.train()
end_time = datetime.now(timezone.utc)
runtime_seconds = (end_time - start_time).total_seconds()
print(f"\nTraining finished in {runtime_seconds:.0f}s")


In [ ]:
# Cell 8: Save artifacts + plot reward curve
import matplotlib.pyplot as plt

os.makedirs("outputs/grpo_v3", exist_ok=True)

reward_rows = []
for idx, log_entry in enumerate(trainer.state.log_history, start=1):
    if "reward" not in log_entry:
        continue
    reward_rows.append({
        "step": int(log_entry.get("step", idx)),
        "reward": float(log_entry.get("reward", 0.0)),
        "reward_std": float(log_entry.get("reward_std", 0.0)),
        "loss": float(log_entry.get("loss", log_entry.get("training_loss", 0.0)) or 0.0),
    })

if not reward_rows:
    print("WARNING: No reward rows captured from trainer logs.")
else:
    csv_path = "outputs/grpo_v3/reward_log.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["step", "reward", "reward_std", "loss"])
        writer.writeheader()
        writer.writerows(reward_rows)
    print(f"Saved reward log: {csv_path} ({len(reward_rows)} rows)")

    rewards = [r["reward"] for r in reward_rows]
    stds = [r["reward_std"] for r in reward_rows]
    steps = list(range(len(rewards)))

    if len(rewards) > 1:
        z = np.polyfit(steps, rewards, 1)
        slope = z[0]
    else:
        slope = 0.0

    summary = {
        "model_size": MODEL_SIZE,
        "num_prompts": NUM_PROMPTS,
        "num_generations": NUM_GENERATIONS,
        "continuation_steps": CONTINUATION_STEPS,
        "confidence_cap": CONFIDENCE_CAP,
        "mean_reward": float(np.mean(rewards)),
        "final_reward": float(rewards[-1]),
        "max_reward": float(max(rewards)),
        "min_reward": float(min(rewards)),
        "mean_reward_std": float(np.mean(stds)),
        "slope": float(slope),
        "runtime_seconds": runtime_seconds,
        "completed_at": end_time.isoformat(),
        "fixes_applied": ["multi_step_rollout", "json_quality_bonus", "confidence_cap_0.7",
                          "same_seed_per_group", "higher_lr_2e-5"],
    }
    with open("outputs/grpo_v3/training_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\n=== Training Summary ===")
    for k, v in summary.items():
        print(f"  {k}: {v}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps, rewards, 'b-', alpha=0.7, label='Reward')
    ax1.fill_between(steps,
                     [r - s for r, s in zip(rewards, stds)],
                     [r + s for r, s in zip(rewards, stds)],
                     alpha=0.2, color='blue', label='+/- 1 std')
    trend = np.poly1d(z)(np.array(steps))
    ax1.plot(steps, trend, 'r--', linewidth=2, label=f'Trend (slope={slope:.4f})')
    ax1.set_xlabel('Training Step')
    ax1.set_ylabel('Reward')
    ax1.set_title(f'GRPO V3.1 Reward Curve - slope={slope:.4f}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(steps, stds, 'g-', alpha=0.7)
    ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Zero (bad)')
    ax2.set_xlabel('Training Step')
    ax2.set_ylabel('Reward Std')
    ax2.set_title('Within-Group Reward Variance (should be > 0)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("outputs/grpo_v3/reward_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nPlot saved: outputs/grpo_v3/reward_curve.png")


In [ ]:
# Cell 9: Eval - run trained model on fresh seeds (multi-step)
print("=== Evaluation: 5 episodes on fresh seeds (multi-step) ===")
FastLanguageModel.for_inference(model)

eval_results = []
for ep in range(5):
    eval_seed = 9000 + ep
    obs = env_reset(seed=eval_seed)
    available = obs.get("available_actions", ["set_primary_endpoint"])
    prompt_text = build_prompt(obs)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True,
                       max_length=preset["seq_len"]).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_COMPLETION_LEN,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

    action = parse_action(response_text, available)
    eval_conf = min(action["confidence"], CONFIDENCE_CAP)
    result = env_step(
        action["action_type"],
        action.get("parameters", {}),
        justification=action.get("justification", ""),
        confidence=eval_conf,
    )
    eval_total = extract_reward(result)

    # Continuation steps for multi-step eval
    for ci in range(CONTINUATION_STEPS):
        if result.get("done", False):
            break
        cont_avail = result.get("observation", {}).get("available_actions", [])
        if not cont_avail:
            break
        result = env_step(cont_avail[ci % len(cont_avail)], confidence=0.3)
        eval_total += extract_reward(result)

    eval_results.append(eval_total)
    print(f"  Ep {ep+1} (seed={eval_seed}): action={action['action_type']} "
          f"conf={eval_conf:.2f} -> multi-step reward={eval_total:.4f}")

print(f"\nEval mean reward: {np.mean(eval_results):.4f} +/- {np.std(eval_results):.4f}")


In [ ]:
# Cell 10: Save model checkpoint
model.save_pretrained("checkpoints/grpo_v3/final")
tokenizer.save_pretrained("checkpoints/grpo_v3/final")
print("Model checkpoint saved to checkpoints/grpo_v3/final")
print("\nDone! Download outputs/grpo_v3/ for your results.")
